In [ ]:
# Step 2: Data Cleaning
import pandas as pd
import os
import re

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import configuration file for paths
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

# Load the raw data file
raw_data_file = os.path.join(config.MAIN_DIR_PATH, config.RAW_DATA_FILE)
df = pd.read_csv(raw_data_file)

Mounted at /content/drive


## **Data Cleaning**

In [ ]:
# Remove unnecessary columns for the text processing task
df = df.drop(columns=['from', 'to', 'date'])

In [ ]:
# Remove duplicate rows based on all columns
df.drop_duplicates(inplace=True)
print(f"DataFrame shape after removing duplicate entries: {df.shape}")

DataFrame shape after removing duplicate entries: (16396445, 3)


In [ ]:
# Identify rows with missing text values
missing_rows = df[df['text'].isnull()]
print(f"\nNumber of rows with missing 'text' entries: {len(missing_rows)}")


Number of rows with missing 'text' entries: 797


In [ ]:
# Group unique conversations by 'folder' and 'dialogueID' to identify incomplete conversations
unique_conversations = df.groupby(['folder', 'dialogueID'])

# Find incomplete conversations where any text is missing within a grouped conversation
incomplete_conversations = unique_conversations.filter(lambda x: x['text'].isnull().any())
print(f"\nNumber of incomplete conversations: {len(incomplete_conversations)}")


Number of incomplete conversations: 8217


In [ ]:
# Remove incomplete conversations based on 'folder' and 'dialogueID'
incomplete_conversations_ids = incomplete_conversations[['folder', 'dialogueID']].drop_duplicates()
df = df[~df.set_index(['folder', 'dialogueID']).index.isin(incomplete_conversations_ids.set_index(['folder', 'dialogueID']).index)]

In [ ]:
# Display the cleaned dataframe's shape and first few rows
print(f"\nDataFrame after removing incomplete conversations. New shape: {df.shape}", "\n")
print(df.head())


DataFrame after removing incomplete conversations. New shape: (16388228, 3)
   folder dialogueID                                               text
0     301      1.tsv   any ideas why java plugin takes so long to load?
1     301      1.tsv                                          java 1.4?
2     301      1.tsv                                                yes
3     301      1.tsv                       java 1.5 loads _much_ faster
4     301      1.tsv  noneus: how can i get 1.5 is there a .deb some...


In [ ]:
# Function to clean text data
def clean_text(text):
    # Email regex pattern to match emails
    email_pat = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    text = text.lower() # Lowercase the text
    text = re.sub(r'http\S+|www\S+', '[URL]', text) # Replace URLs with [URL] placeholder
    text = re.sub(email_pat, '[EMAIL]', text) # Replace email addresses with [EMAIL] placeholder

    # Remove non-alphanumeric characters and keep essential punctuation: ?,.!
    text = re.sub(r'[^a-zA-Z0-9?.!,]+', ' ', text)

    return text.strip()

# Apply the 'clean_text' function to the 'text' column of the dataframe
df['cleaned_text'] = df['text'].apply(clean_text)

<ipython-input-11-cf7c0843679b>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cleaned_text'] = df['text'].apply(clean_text)


In [ ]:
# Save the cleaned data
cleaned_data_file = os.path.join(config.MAIN_DIR_PATH, config.CLEANED_DATA_FILE)
df.to_csv(cleaned_data_file, index=False)
print(f"Cleaned data saved to: {cleaned_data_file}")

Cleaned data saved to: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/cleaned_data.csv
